In [ ]:
import sys, os
# The shared `processing` module lives in the "Segmentation of tumor cells and vessel
# area in spatial transcriptomic data" step folder. Add it to sys.path so the
# `from processing import ...` call below resolves when this notebook is run from its
# own folder.
sys.path.append(os.path.join("..", "3.Segmentation of tumor cells and vessel area in spatial transcriptomic data"))

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

from processing import find_cell_areas, select_specific_cell

In [ ]:
directory = '../data/h5ad_batch/'  # Change this to your target directory path.
file_list = os.listdir(directory)
file_list.sort()

adata_list = []
for file_name in file_list:
    adata = sc.read_h5ad(f'{directory}{file_name}')     
    adata.layers['counts'] = adata.X
    sc.pp.normalize_total(adata, target_sum = 1e3)
    sc.pp.log1p(adata)
    adata.layers['log1p'] = adata.X
    adata_list.append(adata)


# Update the loop to ensure that any COO matrix is converted before further processing
sample_list = [file.split('.')[0] for file in file_list]
adata_list_fin = []

for i, adata in enumerate(adata_list):
    batch_list = adata.obs['batch'].unique().tolist()
    sample_id = sample_list[i]
    for batch in batch_list:
        adata_ = adata[adata.obs['batch'] == batch].copy()
        
        # Process cell areas (this function should work as intended)
        find_cell_areas(adata=adata_)  # Default. cancer_epithelial_filtered
        
        # Convert adata_.X to CSR if it is in COO format to avoid warning issues
        if hasattr(adata_.X, 'format') and adata_.X.format == 'coo':
            adata_.X = adata_.X.tocsr()
        
        adata_list_fin.append(adata_)

In [ ]:
## find vessel area ## 
for adata in adata_list_fin:
    find_cell_areas(adata, cell_type = 'Endothelial_filtered')

In [ ]:
## 2. Calculate the cancer - vessel distance ##

In [ ]:
from scipy.spatial import cKDTree
import numpy as np
import pandas as pd

# Store sample-level distance results.
df_results = []

for adata in adata_list_fin:
    try:
        # Retrieve spatial coordinates and convert pixel units to micrometers.
        coords = adata.obsm["spatial"]
        scalef = (
            adata.uns["spatial"]["Visium_HD"]["scalefactors"]["microns_per_pixel"]
        )
        coords = coords * scalef

        # Extract coordinates of endothelial cells located at vessel boundaries.
        is_edge_endothelial = adata.obs["edge_Endothelial_filtered"]
        edge_endothelial_coords = coords[is_edge_endothelial.values]

        # Distance cannot be calculated if no vessel-edge endothelial cells exist.
        if edge_endothelial_coords.shape[0] == 0:
            print(
                "Warning: No edge_Endothelial_filtered cells were found. "
                "Skipping this sample."
            )
            continue

        # Build a KD-tree from vessel-edge coordinates and calculate the
        # distance from every cell/bin to its nearest vessel boundary.
        tree = cKDTree(edge_endothelial_coords)
        distances, _ = tree.query(coords)
        adata.obs["Distance_from_vessel"] = distances

        # Calculate the mean vessel distance across the entire cancer area.
        if adata.obs["is_Cancer_Epithelial_filtered"].sum() == 0:
            dist_cancer_area = np.nan
            print(
                "Warning: No is_Cancer_Epithelial_filtered cells were found "
                "in this sample."
            )
        else:
            adata_cancer = adata[
                adata.obs["is_Cancer_Epithelial_filtered"]
            ].copy()

            dist_cancer_area = float(
                np.mean(adata_cancer.obs["Distance_from_vessel"])
            )

        # Calculate the mean vessel distance for cancer cells at the tumor edge.
        if adata.obs["edge_Cancer_Epithelial_filtered"].sum() == 0:
            dist_cancer_edge = np.nan
            print(
                "Warning: No edge_Cancer_Epithelial_filtered cells were found "
                "in this sample."
            )
        else:
            adata_cancer_edge = adata[
                adata.obs["edge_Cancer_Epithelial_filtered"]
            ].copy()

            dist_cancer_edge = float(
                np.mean(adata_cancer_edge.obs["Distance_from_vessel"])
            )

        # Use the first unique batch value as the sample identifier.
        batch = adata.obs["batch"].unique().tolist()[0]

        # Save the sample-level results.
        results = [batch, dist_cancer_area, dist_cancer_edge]
        df_results.append(results)

    except Exception as e:
        print(
            f"Error processing sample: {e}. "
            "This sample will be skipped."
        )
        continue

# Convert the collected results into a DataFrame.
df_results = pd.DataFrame(
    df_results,
    columns=[
        "batch",
        "dist_cancer_area",
        "dist_cancer_edge",
    ],
)

In [ ]:
results_df = pd.read_csv('../data/results_df_spatial.csv', index_col = 0)

In [ ]:
results_df = pd.merge(results_df, df_results, on = 'batch')

In [ ]:
df_results_her2 = []
for adata in adata_list_fin:
    select_array = (adata[:,adata.var_names == 'ERBB2'].X.toarray().flatten() != 0) & (np.array(adata.obs['is_Cancer_Epithelial_filtered']))
    adata_sub = adata[select_array].copy()
    dist = float(np.mean(adata_sub.obs['Distance_from_vessel']))
    batch = adata.obs['batch'].unique().tolist()[0]
    results = [batch, dist]
    df_results_her2.append(results)

df_results_her2 = pd.DataFrame(df_results_her2, columns = ['batch', 'dist_her2_cancer'])

In [ ]:
results_df = pd.merge(results_df, df_results_her2, on = 'batch')

In [ ]:
results_df.to_csv('../data/results_df_spatial_vessel.csv')

In [ ]:
#### Save per-fragment figure files ####

In [ ]:
import matplotlib.pyplot as plt

for i in range(len(adata_list_fin)):
    adata_cancer = adata_list_fin[i][adata_list_fin[i].obs['is_Cancer_Epithelial_filtered']].copy()
    batch = adata_cancer.obs['batch'].unique().tolist()[0]
    sc.pl.spatial(adata_cancer, color='Distance_from_vessel', title = f'Distance_from_vessel_{batch}', cmap='jet', vmax=300, show=False)
    plt.savefig(f'../figures/Distance_from_vessel/Distance_from_vessel_{batch}.png', dpi=300)
    plt.close()


In [ ]:
sc.pl.spatial(adata_list_fin[0], color = 'area_Cancer_Epithelial_filtered')

In [ ]:
sc.pl.spatial(adata_list_fin[7], color = 'area_Endothelial_filtered')